# Week 8 — Midterm Review: One End-to-End Pipeline

**Covers:** Weeks 1-7. This notebook is optional and ungraded — a self-check
to confirm all the pieces from the first half of the course connect into a
single pipeline, the way a real project would use them.

We'll use scikit-learn's built-in **breast cancer** dataset one more time, but
this time build the *entire* pipeline in one pass:

1. Load & explore (Week 1)
2. Train/test split + a supervised model (Weeks 2-3)
3. Visualize the data in 2D with PCA (Week 4)
4. Scale features properly (Week 6)
5. Evaluate with cross-validation and a confusion matrix (Week 7)

Some cells are left as **`# TODO`** for you to fill in — try them yourself
before checking the solution further down.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

## Step 1 — Load & explore (Week 1)

In [ ]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target

# TODO: print df.shape, and how many malignant (0) vs benign (1) cases there are
print(df.shape)
print(df["target"].value_counts())

## Step 2 — Visualize with PCA (Week 4)

In [ ]:
X, y = data.data, data.target
X_2d = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(X))

plt.figure(figsize=(6, 5))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap="coolwarm", alpha=0.7, edgecolor="k")
plt.title("Breast Cancer Data in 2D (PCA)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

## Step 3 — Split, scale, and train two models (Weeks 2-3, 6)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

log_reg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
forest = RandomForestClassifier(n_estimators=200, random_state=0)

log_reg.fit(X_train, y_train)
forest.fit(X_train, y_train)

print(f"Logistic regression accuracy: {accuracy_score(y_test, log_reg.predict(X_test)):.2%}")
print(f"Random forest accuracy:       {accuracy_score(y_test, forest.predict(X_test)):.2%}")

## Step 4 — Evaluate properly with cross-validation (Week 7)

In [ ]:
cv_scores_lr = cross_val_score(log_reg, X, y, cv=5)
cv_scores_rf = cross_val_score(forest, X, y, cv=5)

print(f"Logistic regression 5-fold CV: {cv_scores_lr.mean():.2%} (+/- {cv_scores_lr.std():.2%})")
print(f"Random forest 5-fold CV:       {cv_scores_rf.mean():.2%} (+/- {cv_scores_rf.std():.2%})")

In [ ]:
# TODO: plot a confusion matrix for whichever model scored higher on cross-validation
best_model = log_reg if cv_scores_lr.mean() >= cv_scores_rf.mean() else forest
cm = confusion_matrix(y_test, best_model.predict(X_test))
ConfusionMatrixDisplay(cm, display_labels=data.target_names).plot(cmap="Blues")
plt.title("Confusion Matrix — Best Model")
plt.show()

## Self-check questions

Try answering these from memory before looking anything up:

1. Why do we split into train and test sets instead of evaluating on the same
   data we trained on?
2. Why does `StandardScaler` matter more for logistic regression / neural
   networks than for a random forest?
3. What does a point being far from others in the PCA plot suggest about that
   patient's measurements?
4. If cross-validation accuracy is much lower than a single train/test split's
   accuracy, what does that tell you?
5. In this dataset, which mistake is worse — predicting "benign" for an
   actually malignant tumor, or the reverse? Which metric captures that?